# Bronze inventory
Connect to the platform MinIO S3 API as described in README.md. Use the inventory table to explain what landed, then open a collection to show the actual records.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from bronze_review import decode_file, inventory, open_bronze

bronze = open_bronze()
catalog = inventory(bronze)
print(f"Objects listed: {len(bronze.objects)}")

Objects listed: 111


In [2]:
columns = [
    "kind", "source", "collection", "key", "bytes", "rows", "status", "fields",
    "run_id", "published_at", "generation", "partition", "offset", "description",
]
inventory_df = pd.DataFrame.from_records(catalog, columns=columns)
inventory_df.to_csv(Path("bronze_inventory.csv"), index=False, encoding="utf-8-sig")

print("Dataset inventory")
display(inventory_df[["kind", "source", "collection", "rows", "status", "description"]])
exceptions_df = inventory_df[inventory_df["status"].fillna("ok").ne("ok")]
if not exceptions_df.empty:
    print("Inventory exceptions")
    display(exceptions_df[["key", "status"]])

Dataset inventory


,kind,source,collection,rows,status,description
0,batch manifest,dealer-sql,,NaN,NaN,Completed full snapshot.
1,snapshot,dealer-sql,customers,5.0,ok,Dealer customer master.
2,snapshot,dealer-sql,products,5.0,ok,Dealer product master.
3,snapshot,dealer-sql,warehouses,1.0,ok,Dealer stock locations.
4,snapshot,dealer-sql,accounts,5.0,ok,Dealer chart of accounts.
...,...,...,...,...,...,...
106,inbound file,seed-v1,seed-v1/policy/ar-aging-v1.txt,NaN,ok,Seed reference or inbound source file. Inspect...
107,inbound file,seed-v1,seed-v1/policy/reconciliation-v1.txt,NaN,ok,Seed reference or inbound source file. Inspect...
108,inbound file,seed-v1,seed-v1/mdm/account-mappings.json,1.0,ok,Candidate source-to-canonical reference mapping.
109,Kafka delivery,parent.orders.v1,parent.orders.v1,1.0,ok,One raw Kafka delivery with Base64-encoded key...


Batch rows are counted per snapshot and should not be summed to derive unique business records. Kafka rows represent deliveries, each identified by generation, partition, and offset. Unclassified objects remain in MinIO and the inventory for manual inspection.

## Open the actual records
The default example shows the latest dealer customer snapshot. Change the source and collection to show another snapshot, for example `erpnext` and `Sales Invoice`. For an inbound CSV, JSON file, or Kafka envelope, copy its full key from `inventory_df` or `bronze_inventory.csv` into `object_key`. One row represents one source record. Nested ERP line items stay with their parent record.

In [3]:
source = "dealer-sql"
collection = "customers"
object_key = ""

if object_key:
    selected = inventory_df.loc[inventory_df["key"].eq(object_key)]
else:
    snapshots = inventory_df.loc[
        inventory_df["kind"].eq("snapshot") & inventory_df["source"].eq(source)
    ]
    latest_run = snapshots.sort_values(["published_at", "run_id"]).tail(1)["run_id"]
    selected = snapshots.loc[
        snapshots["run_id"].isin(latest_run) & snapshots["collection"].eq(collection)
    ]

if len(selected) != 1:
    raise ValueError("Select a source and collection from inventory_df, or enter one exact object key.")
entry = selected.iloc[0]
if entry["status"] != "ok":
    raise ValueError(f"Choose a readable data object. Selected status: {entry['status']}")
records, fields = decode_file(entry["key"], bronze.read(entry["key"]))
if records is None:
    raise ValueError("Choose a JSON, JSONL, or CSV object to display as a DataFrame.")

records_df = pd.json_normalize(records, max_level=1) if records else pd.DataFrame(columns=fields)
print(f"{entry['source']} | {entry['collection']} | {len(records_df)} records")
print(entry["description"])
display(records_df.head(20))

dealer-sql | customers | 5 records
Dealer customer master.


,id,name,registration_key
0,D-C1,Metro Ind. Services,DEMO-100
1,D-C2,Pioneer Engineering,DEMO-200
2,D-C3,Apex Industrial,None
3,D-C4,Summit Process,None
4,D-C5,Valley Fabrication,DEMO-600
